# Clusterização de Imagens Baseada na Extração de Features

Autor:  Viviane da Rosa Sommer

Atualizado: 22/01/2025

## Objetivo:
Este notebook tem como objetivo aplicar algoritmos de clusterização para agrupar imagens com base nas features extraídas previamente. A clusterização é realizada sobre as representações reduzidas das features, geradas por métodos como PCA, t-SNE e UMAP. Após o treinamento dos algoritmos, o notebook também salva exemplos de imagens pertencentes a cada cluster, permitindo uma análise visual dos agrupamentos.

## Importações Necessárias

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from joblib import dump, load
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import random
import shutil
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pandas as pd
import json
from typing import List, Dict, Tuple, Optional

## Declaração de Constantes e Modelos

In [ ]:
feature_extraction_methods = ["histogram", "hog", "embedding", "swin", "clip"]
dimensionality_reduction_methods = ["pca", "tsne", "umap"]
cluster_range = [3, 5, 7, 9, 15, 20, 25, 30, 50]
n_examples = 5

IMAGE_PATH = "dataset/images"
OUTPUT_FEATURES_PATH = "features"
TRAINED_MODELS_PATH = "trained_models"
os.makedirs(TRAINED_MODELS_PATH, exist_ok=True)

results: List[Dict] = []

## Funções Necessárias

In [ ]:
class NumpyEncoder(json.JSONEncoder):
    """
    Custom encoder to handle NumPy types and arrays for JSON serialization.
    """
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        if isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        return super().default(obj)


def find_image(file_name: str) -> Optional[str]:
    """
    Searches for an image file by name in the 'train' and 'test' subdirectories.

    Args:
        file_name (str): Name of the image file to search for.

    Returns:
        Optional[str]: Full path to the image if found, or None if not found.
    """
    possible_dirs = [os.path.join(IMAGE_PATH, "train"), os.path.join(IMAGE_PATH, "test")]
    for directory in possible_dirs:
        image_path = os.path.join(directory, file_name)
        if os.path.exists(image_path):
            return image_path
    return None


def save_kmeans_model(kmeans_model: KMeans, feature_method: str, reduction_method: str, n_clusters: int) -> None:
    """
    Saves a trained K-Means model to disk.

    Args:
        kmeans_model (KMeans): Trained K-Means model.
        feature_method (str): Feature extraction method used.
        reduction_method (str): Dimensionality reduction method used.
        n_clusters (int): Number of clusters used in the K-Means model.
    """
    model_filename = f"kmeans_{feature_method}_{reduction_method}_{n_clusters}_clusters.joblib"
    model_path = os.path.join(TRAINED_MODELS_PATH, model_filename)
    dump(kmeans_model, model_path)
    print(f"Model saved at: {model_path}")


def train_clustering_with_hyperparameter_search(
    reduction_methods: List[str],
    feature_methods: List[str],
    cluster_range: List[int],
    n_examples: int = 3
) -> None:
    """
    Trains K-Means clustering with various hyperparameter combinations.

    Args:
        reduction_methods (List[str]): List of dimensionality reduction methods.
        feature_methods (List[str]): List of feature extraction methods.
        cluster_range (List[int]): List of cluster counts to test.
        n_examples (int): Number of examples to visualize per cluster.
    """
    global results

    for reduction_method in reduction_methods:
        for feature_method in feature_methods:
            reduced_dir = os.path.join(OUTPUT_FEATURES_PATH, "reduced", reduction_method, "combined", feature_method)
            if not os.path.exists(reduced_dir):
                print(f"Reduced features not found for {feature_method} using {reduction_method}. Skipping...")
                continue

            features: List[np.ndarray] = []
            files: List[str] = []
            for feature_file in os.listdir(reduced_dir):
                if feature_file.endswith(".npy"):
                    file_path = os.path.join(reduced_dir, feature_file)
                    features.append(np.load(file_path))
                    files.append(feature_file)

            if not features:
                print(f"No reduced features found in {reduced_dir}. Skipping...")
                continue

            features = np.array(features)

            for n_clusters in cluster_range:
                print(f"Training K-Means for {feature_method} using {reduction_method} with {n_clusters} clusters...")

                kmeans = KMeans(n_clusters=n_clusters, random_state=42)
                labels = kmeans.fit_predict(features)

                save_kmeans_model(kmeans, feature_method, reduction_method, n_clusters)

                silhouette_avg = silhouette_score(features, labels)
                print(f"Silhouette Score for {feature_method} using {reduction_method} with {n_clusters} clusters: {silhouette_avg:.2f}")

                results.append({
                    "Feature Method": feature_method,
                    "Reduction Method": reduction_method,
                    "n_clusters": n_clusters,
                    "Silhouette Score": silhouette_avg,
                    "files": files,
                    "labels": labels
                })


def save_results_to_files(results: List[Dict], df_results: pd.DataFrame) -> None:
    """
    Saves the clustering results to JSON and CSV files.

    Args:
        results (List[Dict]): List of detailed clustering results.
        df_results (pd.DataFrame): DataFrame summarizing the clustering results.
    """
    results_json_path = "clustering_results.json"
    results_csv_path = "clustering_results.csv"

    with open(results_json_path, "w") as json_file:
        json.dump(results, json_file, indent=4, cls=NumpyEncoder)
    print(f"Detailed results saved to {results_json_path}")

    df_results.to_csv(results_csv_path, index=False)
    print(f"Summarized results saved to {results_csv_path}")


def generate_results_table() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Generates a DataFrame summarizing clustering results and identifies the top 5 results.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: A tuple containing the full results DataFrame and the top 5 results DataFrame.
    """
    global results

    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by="Silhouette Score", ascending=False)
    print("\nComparison of Results:")
    print(df_results)

    top_5_results = df_results.head(5)

    return df_results, top_5_results


def plot_top_kmeans_clusters(top_5_results: pd.DataFrame, n_examples: int = 5) -> None:
    """
    Plots examples from the top 5 K-Means models for each cluster.

    Args:
        top_5_results (pd.DataFrame): DataFrame containing the top 5 clustering results.
        n_examples (int): Number of examples to visualize per cluster.
    """
    for index, row in top_5_results.iterrows():
        feature_method = row["Feature Method"]
        reduction_method = row["Reduction Method"]
        n_clusters = row["n_clusters"]
        files = row["files"]
        labels = row["labels"]

        print(f"\nPlotting examples for {feature_method} + {reduction_method} with {n_clusters} clusters (Silhouette Score: {row['Silhouette Score']:.2f})")

        for cluster_id in range(n_clusters):
            cluster_files = [files[i] for i, label in enumerate(labels) if label == cluster_id]
            selected_files = random.sample(cluster_files, min(len(cluster_files), n_examples))

            plt.figure(figsize=(15, 5))
            for i, file in enumerate(selected_files):
                image_path = find_image(os.path.splitext(file)[0] + ".jpg")
                if image_path:
                    image = Image.open(image_path)
                    plt.subplot(1, len(selected_files), i + 1)
                    plt.imshow(image)
                    plt.title(f"Cluster {cluster_id}")
                    plt.axis("off")
            plt.suptitle(f"{feature_method} + {reduction_method} (n_clusters={n_clusters}) - Cluster {cluster_id}")
            plt.show()

## Clusterização - Execução do K-Means para clusters de tamanho 3, 5, 7, 9, 15

In [ ]:
train_clustering_with_hyperparameter_search(
    reduction_methods=dimensionality_reduction_methods,
    feature_methods=feature_extraction_methods,
    cluster_range=cluster_range,
    n_examples=n_examples
)

## Comparação - Silhouette Score

### O Silhouette Score é uma métrica amplamente utilizada para avaliar a qualidade de agrupamentos em algoritmos de clusterização, como o K-Means. Ele mede o quão bem cada ponto de um dataset está associado ao seu cluster, em comparação com clusters vizinhos. Essa métrica varia de -1 a 1, onde:

- 1.0: O agrupamento é ótimo. Os pontos estão muito próximos ao centro do cluster ao qual pertencem e bem separados de outros clusters.
- 0.0: Os pontos estão no limite entre dois clusters (os clusters não estão bem separados).
- < 0.0: O agrupamento é ruim. Os pontos estão mais próximos de clusters vizinhos do que do cluster ao qual pertencem.

In [ ]:
df_results, top_5_results = generate_results_table()

save_results_to_files(results, df_results)

## Visualização de exemplos de cada cluster, para os 5 melhores K-Means treinados

In [ ]:
plot_top_kmeans_clusters(top_5_results, n_examples=n_examples)